In [ ]:
from pathlib import Path
import os

# Keep relative input and output paths anchored at the project root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "experiments" else Path.cwd()
if not (PROJECT_ROOT / "utils.py").is_file():
    raise RuntimeError(f"Open this notebook from the project root or experiments/: {Path.cwd()}")
os.chdir(PROJECT_ROOT)



In [1]:
import numpy as np
import trimesh
from PIL import Image

def project_masks_to_3d(mesh, masks_data, width=1008, height=1008):
    """
    Projects 2D masks onto a 3D trimesh model using camera parameters.

    Parameters:
        mesh: trimesh.Trimesh object
        masks_data: list or dict containing mask and camera positions.
            Supported formats:
            - List of tuples: [(mask, eye), ...] or [(mask, eye, up), ...]
            - Dict: {name: (mask, eye), ...} or {name: (mask, eye, up), ...}
            - Dict of dicts: {name: {"mask": mask, "eye": eye, "up": up}, ...}
        width, height: Resolution of the rendered views (matching mask dimensions)

    Returns:
        colored_mesh: trimesh.Trimesh with green highlighted masked vertices
        mask_submesh: trimesh.Trimesh submesh of the masked region
    """
    vertices = mesh.vertices
    normals = mesh.vertex_normals

    # Calculate bounding box parameters natively in Trimesh
    bounds = mesh.bounds
    center = bounds.mean(axis=0)
    extent = mesh.extents.max()

    half_size = (extent / 2.0) * 1.2
    aspect = width / height

    vertex_masked = np.zeros(len(vertices), dtype=bool)
    rel_verts = vertices - center

    # Unify masks_data into a list of tuples: (mask, eye, up)
    items = []
    if isinstance(masks_data, dict):
        for val in masks_data.values():
            if isinstance(val, dict):
                mask = val["mask"]
                eye = np.array(val["eye"], dtype=float)
                up = np.array(val["up"], dtype=float) if "up" in val and val["up"] is not None else None
            elif isinstance(val, (tuple, list)):
                mask = val[0]
                eye = np.array(val[1], dtype=float)
                up = np.array(val[2], dtype=float) if len(val) > 2 else None
            items.append((mask, eye, up))
    elif isinstance(masks_data, (list, tuple)):
        for val in masks_data:
            if isinstance(val, dict):
                mask = val["mask"]
                eye = np.array(val["eye"], dtype=float)
                up = np.array(val["up"], dtype=float) if "up" in val and val["up"] is not None else None
            elif isinstance(val, (tuple, list)):
                mask = val[0]
                eye = np.array(val[1], dtype=float)
                up = np.array(val[2], dtype=float) if len(val) > 2 else None
            items.append((mask, eye, up))

    for mask_img, eye, up in items:
        # Convert mask to boolean numpy array
        if isinstance(mask_img, Image.Image):
            mask_arr = np.array(mask_img) > 128
        else:
            mask_arr = mask_img > 0

        # Camera orientation basis
        z_cam = (eye - center) / np.linalg.norm(eye - center)

        # Auto-detect 'up' vector if missing
        if up is None:
            if np.abs(z_cam[1]) > 0.99:  # Top or Bottom view
                up = np.array([0.0, 0.0, -1.0]) if z_cam[1] > 0 else np.array([0.0, 0.0, 1.0])
            else:
                up = np.array([0.0, 1.0, 0.0])

        x_cam = np.cross(up, z_cam)
        x_cam /= np.linalg.norm(x_cam)
        y_cam = np.cross(z_cam, x_cam)
        y_cam /= np.linalg.norm(y_cam)

        # 1. Back-face culling
        visible_mask = np.dot(normals, z_cam) > 0.1

        # 2. Project 3D coordinates to NDC
        x_proj = np.dot(rel_verts, x_cam)
        y_proj = np.dot(rel_verts, y_cam)

        u_ndc = x_proj / (half_size * aspect)
        v_ndc = y_proj / half_size

        # Convert NDC to pixel indices
        px = np.clip(np.round((u_ndc + 1.0) / 2.0 * (width - 1)).astype(int), 0, width - 1)
        py = np.clip(np.round((1.0 - v_ndc) / 2.0 * (height - 1)).astype(int), 0, height - 1)

        # 3. Sample mask
        sampled_mask = mask_arr[py, px]

        # Accumulate mask flags
        vertex_masked |= (visible_mask & sampled_mask)

    # --- Color 3D Mesh ---
    colored_mesh = mesh.copy()
    colors = np.full((len(vertices), 4), [180, 180, 180, 255], dtype=np.uint8)
    colors[vertex_masked] = [0, 255, 0, 255]
    colored_mesh.visual.vertex_colors = colors

    # --- Extract Mask Submesh ---
    masked_faces = vertex_masked[mesh.faces].any(axis=1)
    mask_submesh = mesh.submesh([masked_faces], append=True)

    return colored_mesh, mask_submesh